# Leaf Pipeline Playground

这个 notebook 用来测试新的三段式叶子流程：`原始材料 -> 抽取 items -> 路由到叶子 -> 可选生成卡片`。

设计原则：
- 默认只往 `scratch/leaf_agent_playground/` 写测试结果
- 不自动改正式知识树
- 先看抽取和路由，再决定是否进入卡片生成


In [1]:
%pip install -U azure-ai-projects azure-identity openai nest_asyncio pydantic



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/anaconda3/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from material_extractor import (
    FoundryMaterialExtractor,
    MaterialExtractionParseError,
    material_extraction_environment,
    summarize_result as summarize_extraction,
)
from leaf_router import (
    FoundryLeafRouter,
    LeafRoutingParseError,
    leaf_routing_environment,
    summarize_result as summarize_routing,
)
from leaf_card_agent import (
    DraftPayloadParseError,
    FoundryLeafCardDraftAgent,
    leaf_card_environment,
    summarize_result as summarize_cards,
)

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'leaf_agent_playground'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTED_JSON = SCRATCH_DIR / 'pipeline_extracted.json'
EXTRACTED_ROUGH_JSON = SCRATCH_DIR / 'pipeline_extracted_rough.json'
EXTRACTED_RAW_TXT = SCRATCH_DIR / 'pipeline_extracted_raw.txt'

ROUTES_JSON = SCRATCH_DIR / 'pipeline_routes.json'
ROUTES_ROUGH_JSON = SCRATCH_DIR / 'pipeline_routes_rough.json'
ROUTES_RAW_TXT = SCRATCH_DIR / 'pipeline_routes_raw.txt'

CARDS_ROUGH_JSON = SCRATCH_DIR / 'pipeline_cards_rough.json'
CARDS_NORMALIZED_JSON = SCRATCH_DIR / 'pipeline_cards_normalized.json'
CARDS_DRAFT_MD = SCRATCH_DIR / 'pipeline_cards_draft.md'
CARDS_JSONL = SCRATCH_DIR / 'pipeline_cards.jsonl'


## 1. 环境检查

如果这里 deployment、endpoint 或 `structured_outputs_supported` 看起来不对，先修环境再往下跑。


In [3]:
print('extractor:', json.dumps(material_extraction_environment(), ensure_ascii=False, indent=2))
print('router:', json.dumps(leaf_routing_environment(), ensure_ascii=False, indent=2))
print('card_agent:', json.dumps(leaf_card_environment(), ensure_ascii=False, indent=2))


extractor: {
  "project_endpoint": "https://teachagent-xumuchi-20260614.services.ai.azure.com/api/projects/proj-default",
  "model_deployment": "gpt-4o-mini",
  "structured_outputs_supported": "True"
}
router: {
  "project_endpoint": "https://teachagent-xumuchi-20260614.services.ai.azure.com/api/projects/proj-default",
  "model_deployment": "gpt-4o-mini",
  "structured_outputs_supported": "True"
}
card_agent: {
  "project_endpoint": "https://teachagent-xumuchi-20260614.services.ai.azure.com/api/projects/proj-default",
  "model_deployment": "gpt-4o-mini",
  "structured_outputs_supported": "True"
}


## 2. 准备一段原始材料

你可以直接把 OCR 结果、讲义段落、笔记文字贴进来。先不要写 `node_id`。


In [4]:
raw_text = '''
极值点偏移问题的七种解法
极值点偏移是导数题目中常见的一大类问题，尤其在选择填空和导数大题中经常出现。解决这类问题的关键在于简化已知函数，挖掘隐含条件。以下是七种常用的解法：
构造对数不等式
如果函数有两个零点，且在某点处导数小于零，则该点处函数值小于零。通过构造对数不等式，可以比较几何中的函数值与零点的大小关系。
三阶导法与构造对称
对于某些题目，构造对数不等式可能无法直接提取因式，此时可以考虑使用三阶导法。通过构造对称函数，可以简化计算过程。
对数均值不等式
对数均值不等式是一种常用的方法，尤其在解决极值点偏移问题时非常有效。通过比较几何中的函数值与零点的大小关系，可以得出结论。
比差值代换法
对于某些题目，直接使用对数不等式可能无法解决问题，此时可以考虑使用比差值代换法。通过构造函数并比较其大小关系，可以得出结论。
单调性法
如果函数在某点处单调递减，且该点处函数值大于零，则该点处导数小于零。通过单调性法，可以比较几何中的函数值与零点的大小关系。
中点切线斜率法
对于某些题目，通过构造中点切线斜率法可以简化计算过程。通过比较函数值与零点的大小关系，可以得出结论。
双变量法
对于某些题目，双变量法是一种有效的解决方法。通过构造双变量函数，可以简化计算过程，并得出正确的结论。
'''.strip()

print(raw_text[:600])


极值点偏移问题的七种解法
极值点偏移是导数题目中常见的一大类问题，尤其在选择填空和导数大题中经常出现。解决这类问题的关键在于简化已知函数，挖掘隐含条件。以下是七种常用的解法：
构造对数不等式
如果函数有两个零点，且在某点处导数小于零，则该点处函数值小于零。通过构造对数不等式，可以比较几何中的函数值与零点的大小关系。
三阶导法与构造对称
对于某些题目，构造对数不等式可能无法直接提取因式，此时可以考虑使用三阶导法。通过构造对称函数，可以简化计算过程。
对数均值不等式
对数均值不等式是一种常用的方法，尤其在解决极值点偏移问题时非常有效。通过比较几何中的函数值与零点的大小关系，可以得出结论。
比差值代换法
对于某些题目，直接使用对数不等式可能无法解决问题，此时可以考虑使用比差值代换法。通过构造函数并比较其大小关系，可以得出结论。
单调性法
如果函数在某点处单调递减，且该点处函数值大于零，则该点处导数小于零。通过单调性法，可以比较几何中的函数值与零点的大小关系。
中点切线斜率法
对于某些题目，通过构造中点切线斜率法可以简化计算过程。通过比较函数值与零点的大小关系，可以得出结论。
双变量法
对于某些题目，双变量法是一种有效的解决方法。通过构造双变量函数，可以简化计算过程，并得出正确的结论。


## 3. 先做素材抽取

这一步只负责拆成 `items[]`，不绑定知识树。


In [5]:
extractor = FoundryMaterialExtractor(max_tokens=2600, temperature=0.1)

extraction_result = None
try:
    extraction_result = extractor.generate(
        material_text=raw_text,
        extra_rule='尽量把每一种独立方法拆成单独 item；标题直接写方法名，不要写“第几种方法”。',
    )
    print(json.dumps(summarize_extraction(extraction_result), ensure_ascii=False, indent=2))
except MaterialExtractionParseError as exc:
    print(str(exc))
    print('\n===== RAW MODEL OUTPUT =====\n')
    print(exc.raw_output)


{
  "response_source": "ai_tool_json_mode",
  "document_title": "极值点偏移问题的七种解法",
  "document_summary": "本文列举并简要说明了解决极值点偏移类导数题的七种常用方法，包括构造对数不等式、三阶导法与构造对称、对数均值不等式、比差值代换法、单调性法、中点切线斜率法以及双变量法。",
  "global_keywords": [
    "极值点偏移",
    "导数",
    "不等式",
    "方法",
    "函数零点"
  ],
  "source_quality_notes": [
    "材料为概述性文本，未标明具体来源或作者，适合作为教学素材的初步整理。"
  ],
  "unresolved_points": [],
  "item_count": 7,
  "item_titles": [
    "构造对数不等式",
    "三阶导法与构造对称",
    "对数均值不等式",
    "比差值代换法",
    "单调性法",
    "中点切线斜率法",
    "双变量法"
  ]
}


## 4. 查看抽取结果并写入 scratch


In [7]:
if extraction_result is not None:
    EXTRACTED_JSON.write_text(
        json.dumps(extraction_result.normalized_payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    EXTRACTED_ROUGH_JSON.write_text(
        json.dumps(extraction_result.rough_payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    EXTRACTED_RAW_TXT.write_text(extraction_result.raw_model_output, encoding='utf-8')
    print(f'wrote: {EXTRACTED_JSON}')
    print(f'wrote: {EXTRACTED_ROUGH_JSON}')
    print(f'wrote: {EXTRACTED_RAW_TXT}')
    print(json.dumps(extraction_result.normalized_payload, ensure_ascii=False, indent=2))


wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_extracted.json
wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_extracted_rough.json
wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_extracted_raw.txt
{
  "document_title": "极值点偏移问题的七种解法",
  "document_summary": "本文列举并简要说明了解决极值点偏移类导数题的七种常用方法，包括构造对数不等式、三阶导法与构造对称、对数均值不等式、比差值代换法、单调性法、中点切线斜率法以及双变量法。",
  "global_keywords": [
    "极值点偏移",
    "导数",
    "不等式",
    "方法",
    "函数零点"
  ],
  "source_quality_notes": [
    "材料为概述性文本，未标明具体来源或作者，适合作为教学素材的初步整理。"
  ],
  "unresolved_points": [],
  "items": [
    {
      "kind_guess": "method",
      "title": "构造对数不等式",
      "source_excerpt": "如果函数有两个零点，且在某点处导数小于零，则该点处函数值小于零。通过构造对数不等式，可以比较几何中的函数值与零点的大小关系。",
      "summary": "当函数有两个零点且在某点处导数为负时，可通过构造对数不等式比较该点的函数值与零点的大小，从而判断极值点的偏移。",
      "keywords": [
        "对数不等式",
        "函数零点",
        "导数符号",
        "极值点偏移"
      ],
      "item_id": "item_01",
      "alias

## 5. 做叶子路由

这一步会先生成候选叶子 / 候选父节点，再让模型在候选里判断：
- `match_existing`
- `create_new_leaf`
- `uncertain`


In [8]:
router = FoundryLeafRouter(max_tokens=2800, temperature=0.1)

routing_result = None
if extraction_result is not None:
    try:
        routing_result = router.generate(
            extracted_payload=extraction_result.normalized_payload,
            extra_rule='优先挂到导数专题下的已有叶子；确实不合适时再提出新叶子。',
            candidate_limit=6,
            parent_limit=5,
        )
        print(json.dumps(summarize_routing(routing_result), ensure_ascii=False, indent=2))
    except LeafRoutingParseError as exc:
        print(str(exc))
        print('\n===== RAW MODEL OUTPUT =====\n')
        print(exc.raw_output)


{
  "response_source": "ai_tool_minimal_text_json",
  "document_title": null,
  "routing_notes": [],
  "route_count": 7,
  "decisions": [
    {
      "item_id": "item_01",
      "title": "构造对数不等式",
      "decision": "match_existing",
      "matched_node_id": "math.数列与不等式.不等式.简单线性规划求解",
      "parent_node_id": null,
      "proposed_name": null
    },
    {
      "item_id": "item_02",
      "title": "三阶导法与构造对称",
      "decision": "match_existing",
      "matched_node_id": "math.数列与不等式.不等式.简单线性规划求解",
      "parent_node_id": null,
      "proposed_name": null
    },
    {
      "item_id": "item_03",
      "title": "对数均值不等式",
      "decision": "match_existing",
      "matched_node_id": "math.数列与不等式.不等式.简单线性规划求解",
      "parent_node_id": null,
      "proposed_name": null
    },
    {
      "item_id": "item_04",
      "title": "比差值代换法",
      "decision": "match_existing",
      "matched_node_id": "math.数列与不等式.不等式.简单线性规划求解",
      "parent_node_id": null,
      "proposed_name": null
    },
    {

## 6. 查看路由结果并写入 scratch


In [9]:
if routing_result is not None:
    ROUTES_JSON.write_text(
        json.dumps(routing_result.normalized_payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    ROUTES_ROUGH_JSON.write_text(
        json.dumps(routing_result.rough_payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    ROUTES_RAW_TXT.write_text(routing_result.raw_model_output, encoding='utf-8')
    print(f'wrote: {ROUTES_JSON}')
    print(f'wrote: {ROUTES_ROUGH_JSON}')
    print(f'wrote: {ROUTES_RAW_TXT}')
    print(json.dumps(routing_result.normalized_payload, ensure_ascii=False, indent=2))


wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_routes.json
wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_routes_rough.json
wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_routes_raw.txt
{
  "document_title": null,
  "routing_notes": [],
  "routes": [
    {
      "item_id": "item_01",
      "title": "构造对数不等式",
      "decision": "match_existing",
      "matched_node_id": "math.数列与不等式.不等式.简单线性规划求解",
      "proposed_name": null,
      "proposed_kind": null,
      "parent_node_id": null,
      "confidence": null,
      "reason": "模型未提供原因。",
      "candidate_node_ids": [
        "math.数列与不等式.不等式.简单线性规划求解",
        "math.集合_映射_函数与导数.导数.利用导数判断单调性",
        "math.数列与不等式.不等式.一元二次不等式解法",
        "math.集合_映射_函数与导数.导数.导数解决生活优化问题",
        "math.集合_映射_函数与导数.导数.利用导数求函数极值",
        "math.集合_映射_函数与导数.导数.利用导数求函数最值"
      ],
      "candidate_parent_node_ids": [
        "math.数列与不等式.不等式",
        "math.集合_映射

## 7. 手动确认哪些 node_id 要进入卡片生成

这里只默认选 `decision == 'match_existing'` 的结果。
如果你想手工改，可以直接编辑 `confirmed_node_ids`。


In [10]:
confirmed_node_ids = []
if routing_result is not None:
    confirmed_node_ids = [
        route['matched_node_id']
        for route in routing_result.routes
        if route['decision'] == 'match_existing' and route.get('matched_node_id')
    ]

confirmed_node_ids = list(dict.fromkeys(confirmed_node_ids))
print(json.dumps(confirmed_node_ids, ensure_ascii=False, indent=2))


[
  "math.数列与不等式.不等式.简单线性规划求解"
]


## 8. 可选：生成叶子卡片

只有当你认可上一步的 `confirmed_node_ids` 时，再跑这一格。
这一步仍然只写入 `scratch/leaf_agent_playground/`。


In [11]:
card_agent = FoundryLeafCardDraftAgent(max_tokens=2600, temperature=0.1)

card_result = None
if confirmed_node_ids:
    try:
        card_result = card_agent.generate(
            node_ids=confirmed_node_ids,
            reference_text=raw_text,
            extra_rule='优先写方法卡；如果材料不足，可以基于高中数学常规知识做克制补全。',
            source='leaf_pipeline_playground',
        )
        print(json.dumps(summarize_cards(card_result), ensure_ascii=False, indent=2))
    except DraftPayloadParseError as exc:
        print(str(exc))
        print('\n===== RAW MODEL OUTPUT =====\n')
        print(exc.raw_output)
else:
    print('当前没有 confirmed_node_ids，先检查路由结果。')


{
  "response_source": "ai_tool_json_mode",
  "document_title": "简单线性规划求解方法卡",
  "document_summary": "介绍在高中不等式章节中，利用线性规划方法求解线性目标函数在线性约束下的最大值或最小值的完整步骤。",
  "global_keywords": [
    "线性规划",
    "不等式",
    "最值",
    "可行域",
    "单纯形法"
  ],
  "source_quality_notes": [
    "参考材料主要涉及极值点偏移问题，未直接提供线性规划内容，依据高中数学教材补全。"
  ],
  "unresolved_points": [],
  "card_count": 1,
  "node_ids": [
    "math.数列与不等式.不等式.简单线性规划求解"
  ]
}


## 9. 查看卡片结果并写入 scratch


In [12]:
if card_result is not None:
    CARDS_ROUGH_JSON.write_text(
        json.dumps(card_result.rough_payload, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    CARDS_NORMALIZED_JSON.write_text(
        json.dumps(card_result.entries, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    CARDS_DRAFT_MD.write_text(card_result.draft_markdown, encoding='utf-8')
    with CARDS_JSONL.open('w', encoding='utf-8') as fp:
        for row in card_result.validated_cards:
            fp.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f'wrote: {CARDS_ROUGH_JSON}')
    print(f'wrote: {CARDS_NORMALIZED_JSON}')
    print(f'wrote: {CARDS_DRAFT_MD}')
    print(f'wrote: {CARDS_JSONL}')
    print(card_result.draft_markdown)


wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_cards_rough.json
wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_cards_normalized.json
wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_cards_draft.md
wrote: /Users/xumuchi/Desktop/TeachAgent/scratch/leaf_agent_playground/pipeline_cards.jsonl
### math.数列与不等式.不等式.简单线性规划求解
- card_type: method_card
- title: 简单线性规划求解
- keywords: 线性规划 | 不等式约束 | 目标函数 | 可行域 | 极值
- aliases: 线性规划求解 | 线性规划方法
- method_goal: 在满足线性不等式约束的条件下，求目标函数的最大值或最小值。
- trigger_signals: 目标函数为线性表达式且约束为线性不等式 | 需要在多约束下求最值 | 出现系数矩阵形式的约束
- applicable_problem_types: 求最大值问题 | 求最小值问题 | 资源分配问题
- steps: 写出目标函数，例如 Z = c1·x + c2·y | 列出所有线性不等式约束并标明方向 | 绘制约束对应的可行域（若变量非负，加入 x≥0, y≥0） | 求出可行域的所有顶点（交点） | 将每个顶点代入目标函数，计算 Z 的值 | 比较得到的 Z 值，选取最大值或最小值对应的点
- failure_modes: 忽略变量非负条件导致可行域错误 | 漏掉约束交点导致顶点不全 | 在目标函数计算时符号或系数写错 | 把不等式方向写反，导致可行域相反 | 可行域为空却仍求解
- review_cue: 线性规划求最值，只需比较目标函数在可行域顶点的值。
- tags: 不等式 | 数列与不等式

## 10. 快速回看 scratch 中的结果文件


In [13]:
for path in sorted(SCRATCH_DIR.iterdir()):
    print(path.name)


offset_methods_draft.md
offset_methods_from_rough_cards.jsonl
offset_methods_from_rough_draft.md
offset_methods_normalized.json
offset_methods_rough.json
pipeline_cards.jsonl
pipeline_cards_draft.md
pipeline_cards_normalized.json
pipeline_cards_rough.json
pipeline_extracted.json
pipeline_extracted_raw.txt
pipeline_extracted_rough.json
pipeline_routes.json
pipeline_routes_raw.txt
pipeline_routes_rough.json
